# Passim : repérage des citations bibliques

Ce notebook Google Colab exécute un pipeline complet :
1. installer Passim ;
2. charger les textes cibles depuis des fichiers `.txt` ;
3. charger la Bible comme corpus de référence ;
4. lancer Passim ;
5. vérifier les prédictions à l'aide du gold CSV ;
6. exporter chaque exécution dans un dossier séparé avec les paramètres utilisés.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Svetlana-Yatsyk/passim_atelier_2026/blob/main/notebooks/passim_colab.ipynb)

In [ ]:
!pip install pandas tqdm --quiet

from pathlib import Path
import subprocess

import pandas as pd
import json
import os
import re
import shutil
import subprocess
import sys
import unicodedata
from bisect import bisect_left
from datetime import datetime
from html import escape

from IPython.display import HTML, display

In [4]:
REPO_DIR = Path("/content/passim_atelier")

if "google.colab" in str(get_ipython()):
    if not REPO_DIR.exists():
        !git clone https://github.com/Svetlana-Yatsyk/passim_atelier_2026.git {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"Dossier courant : {Path.cwd()}")

Cloning into '/content/passim_atelier'...
remote: Enumerating objects: 1157, done.
remote: Counting objects: 100% (1157/1157), done.
remote: Compressing objects: 100% (1133/1133), done.
remote: Total 1157 (delta 8), reused 1154 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (1157/1157), 3.71 MiB | 20.43 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Dossier courant : /content/passim_atelier


## 1. Installation de Passim et des dépendances

In [5]:
# Passim s'appuie sur Apache Spark, qui a besoin de Java.

result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print('Java détectée :' if result.returncode == 0 else 'Java absente :', result.stderr.splitlines()[0] if result.stderr else 'non détectée')

# Si Java n'est pas disponible dans Colab, décommentez les lignes ci-dessous :
# !apt-get install -y openjdk-11-jdk-headless -qq
# import os
# os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

Java détectée : openjdk version "17.0.17" 2025-10-21


In [6]:
!pip install git+https://github.com/dasmiq/passim.git@seriatim

  Cloning https://github.com/dasmiq/passim.git (to revision seriatim) to /tmp/pip-req-build-gmzk3ca1
  Running command git clone --filter=blob:none --quiet https://github.com/dasmiq/passim.git /tmp/pip-req-build-gmzk3ca1
  Running command git checkout -b seriatim --track origin/seriatim
  Switched to a new branch 'seriatim'
  Branch 'seriatim' set up to track remote branch 'seriatim' from 'origin'.
  Resolved https://github.com/dasmiq/passim.git to commit 0887e9481744c610d9d3ed65285b496196e4a2a3
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.3 MB/s eta 0:00:00
  Created wheel for passim: filename=passim-2.0.0a2-py3-none-any.whl size=14510 sha256=1e547966431196e6e833b559a779421ac10b56b28c8f316c45006b9dec2a031e
  Stored in directory: /tmp/pip-ephem-wheel-cache-61oec58b/wheels/cd/29/e5/cc431424a14f8587a899407713b0681e6a4e2b91de6ea16c4d
Successfully built passim


In [7]:
import passim

In [8]:
!passim --help

https://repos.spark-packages.org/ added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ed3b60e1-6c9a-499b-ac4e-164dfd325221;1.0
	confs: [default]
	found graphframes#graphframes;0.8.0-spark3.0-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
downloading https://repos.spark-packages.org/graphframes/graphframes/0.8.0-spark3.0-s_2.12/graphframes-0.8.0-spark3.0-s_2.12.jar ...
	[SUCCESSFUL ] graphframes#graphframes;0.8.0-spark3.0-s_2.12!graphframes.jar (87ms)
downloading https://repo1.maven.org/maven2/org/slf4j/slf4j-api/1.7.16/slf4j-api-1.7.16.jar ...
	[SUCCESSFUL ] org.slf4j#slf4j-api;1.7.16!slf4j-api.jar (51

## 2. Chargement des textes à interroger

La Bible reste toujours le corpus de référence. Les citations sont recherchées uniquement dans :
- des fichiers `.txt` placés dans `data/txt/`.

Le `doc_id` est conservé tel qu'il apparaît dans le nom du fichier afin de pouvoir évaluer les prédictions contre le gold CSV.

In [9]:
# ============================================================
# OUTILS DE CHARGEMENT DES DOCUMENTS SOURCE
# ============================================================
from google.colab import files


def parse_txt_filename_metadata(filename):
    stem = Path(filename).stem
    parts = stem.split('_', 2)
    source = stem.split('-', 1)[0].strip() if '-' in stem else stem.strip()
    ref_book = parts[1].strip() if len(parts) > 1 else None
    ref_verse = parts[2].strip().replace('_', ':') if len(parts) > 2 else None
    return {
        'filename': filename,
        'stem': stem,
        'source': source or stem,
        'REF_book': ref_book,
        'REF_verse': ref_verse,
    }


def normalize_text(text):
    text = str(text or '').lower()
    #text = ''.join(' ' if unicodedata.category(char).startswith('P') else char for char in text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def build_document_record(name, text, origin):
    meta = parse_txt_filename_metadata(name)
    return {
        'doc_id': meta['stem'],
        'source': meta['source'],
        'source_filename': name,
        'source_origin': origin,
        'text': normalize_text(text),
    }


def load_documents_from_txt_dir(txt_dir):
    txt_dir = Path(txt_dir)
    if not txt_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable : {txt_dir}")
    documents = [
        build_document_record(path.name, path.read_text(encoding='utf-8'), origin=f'dossier:{txt_dir}')
        for path in sorted(txt_dir.glob('*.txt'))
    ]
    if not documents:
        raise FileNotFoundError(f"Aucun fichier .txt dans {txt_dir}")
    return documents


def load_documents_from_upload():
    if files is None:
        raise RuntimeError("L'import manuel n'est disponible que dans Google Colab.")
    uploaded = files.upload()
    documents = [
        build_document_record(name, data.decode('utf-8'), origin='upload_colab')
        for name, data in uploaded.items()
    ]
    if not documents:
        raise ValueError("Aucun fichier n'a été importé.")
    return documents


def summarize_documents(documents):
    pd.DataFrame([
        {
            'doc_id': doc['doc_id'],
            'source': doc['source'],
            'chars': len(doc['text']),
            'origin': doc['source_origin'],
        }
        for doc in documents
    ])

In [10]:
TXT_DIR = Path('data/txt')

YOUR_TEXTS = load_documents_from_txt_dir(TXT_DIR)

print(f"Documents source chargés : {len(YOUR_TEXTS)}")
summarize_documents(YOUR_TEXTS)

Documents source chargés : 1123


## 3. Conversion des textes source en JSONL

Les documents source sont découpés en chunks. Par défaut, le notebook utilise un fenêtrage par phrases (`sentence_window`).

In [11]:
# ============================================================
# PARAMÈTRES DE CHUNKING
# ============================================================
CHUNK_MODE = 'sentence_window'   # 'sentence_window', 'sentence', 'char', 'full'
SENTENCES_PER_CHUNK = 2
SENTENCE_STRIDE = 1
CHUNK_SIZE = 300
CHUNK_OVERLAP = 80
MIN_CHUNK_CHARS = 20
# ============================================================

In [12]:
def chunk_text(
    text,
    mode=CHUNK_MODE,
    sentences_per_chunk=SENTENCES_PER_CHUNK,
    sentence_stride=SENTENCE_STRIDE,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
    min_chars=MIN_CHUNK_CHARS,
):
    """Découpe un texte en chunks et conserve les offsets originaux."""
    text = str(text or '')
    if not text.strip():
        return []

    mode = str(mode or 'sentence_window').strip().lower()

    if mode in {'full', 'none', 'document', 'whole'}:
        piece = text.strip()
        if len(piece) < min_chars:
            return []
        start = text.find(piece)
        return [{'chunk_index': 0, 'text': piece, 'start': start, 'end': start + len(piece)}]

    if mode == 'char':
        step = max(1, chunk_size - overlap)
        chunks = []
        for i, pos in enumerate(range(0, len(text), step)):
            raw_piece = text[pos: pos + chunk_size]
            piece = raw_piece.strip()
            if len(piece) < min_chars:
                continue
            left_trim = len(raw_piece) - len(raw_piece.lstrip())
            start = pos + left_trim
            end = start + len(piece)
            chunks.append({'chunk_index': i, 'text': piece, 'start': start, 'end': end})
        return chunks

    sentences = [s.strip() for s in re.split(r'(?<=[.!?;:])\s+', text) if s.strip()]
    if not sentences:
        piece = text.strip()
        if len(piece) < min_chars:
            return []
        start = text.find(piece)
        return [{'chunk_index': 0, 'text': piece, 'start': start, 'end': start + len(piece)}]

    if mode == 'sentence':
        pieces = sentences
    elif mode == 'sentence_window':
        win = max(1, int(sentences_per_chunk))
        step = max(1, int(sentence_stride))
        pieces = []
        for i in range(0, len(sentences), step):
            piece = ' '.join(sentences[i:i + win]).strip()
            if piece:
                pieces.append(piece)
    else:
        raise ValueError("CHUNK_MODE doit être 'sentence_window', 'sentence', 'char' ou 'full'.")

    chunks = []
    cursor = 0
    for i, piece in enumerate(pieces):
        if len(piece) < min_chars:
            continue
        start = text.find(piece, cursor)
        if start < 0:
            start = text.find(piece)
        if start < 0:
            continue
        end = start + len(piece)
        cursor = max(cursor, start)
        chunks.append({'chunk_index': i, 'text': piece, 'start': start, 'end': end})
    return chunks


def texts_to_jsonl_records(documents):
    records = []
    chunk_meta = {}

    for doc in documents:
        for chunk in chunk_text(doc['text']):
            chunk_id = f"source::{doc['doc_id']}::chunk::{chunk['chunk_index']}"
            record = {
                'id': chunk_id,
                'group': 'source',
                'doc_id': doc['doc_id'],
                'source': doc['source'],
                'source_filename': doc['source_filename'],
                'source_origin': doc['source_origin'],
                'chunk_index': chunk['chunk_index'],
                'chunk_start': chunk['start'],
                'chunk_end': chunk['end'],
                'text': chunk['text'],
            }
            records.append(record)
            chunk_meta[chunk_id] = {
                'doc_id': doc['doc_id'],
                'source': doc['source'],
                'source_filename': doc['source_filename'],
                'source_origin': doc['source_origin'],
                'chunk_index': chunk['chunk_index'],
                'text': chunk['text'],
            }
    return records, chunk_meta

In [13]:
task_records, chunk_meta = texts_to_jsonl_records(YOUR_TEXTS)

print(f'Mode de chunking : {CHUNK_MODE}')
print(f'Chunks source créés : {len(task_records)}')
if task_records:
    print('\nExemple de chunk :')
    print(json.dumps(task_records[0], indent=2, ensure_ascii=False))

Mode de chunking : sentence_window
Chunks source créés : 1123

Exemple de chunk :
{
  "id": "source::gregorianum-p-00nL37E5::chunk::0",
  "group": "source",
  "doc_id": "gregorianum-p-00nL37E5",
  "source": "gregorianum",
  "source_filename": "gregorianum-p-00nL37E5.txt",
  "source_origin": "dossier:data/txt",
  "chunk_index": 0,
  "chunk_start": 0,
  "chunk_end": 931,
  "text": "sciendum uero nobis est quia nequaquam culmen contemplationis attingimus si non ab exterioris cure oppressione cessemus nequaquam nosmetipsos intuemur ut sciamus aliud in nobis rationale qui edificant sibi solitudines iob iii 14 bis solitudines quippe edificare est a secreto cordis terrenorum desideriorum tumultus expellere et una intentione eterne patrie in amore intime quietis anhelare annon cunctos a se cogitationum tumultus expulerat qui dicebat unam petii a domino hanc requiram ut inhabitem in domo domini omnibus diebus uite mee psal xxvi 4 frequentia quippe terrenorum desideriorum fugerat et ad magnam ui

## 4. Chargement de la Bible et conversion en JSONL

La Bible est toujours le texte de référence. Chaque verset est stocké comme une entrée indépendante dans la serie `bible`.

In [ ]:
# ============================================================
# CHARGEMENT DU CSV BIBLIQUE
# Colonnes attendues : book, chapter, verse, latin
# ============================================================
BIBLE_CSV_PATH = "data/csv/clem_vulgate.csv"

bible_df = pd.read_csv(BIBLE_CSV_PATH)

# Si on veut garder uniquement les Psaumes
bible_df = bible_df[bible_df["book"] == "Ps"].reset_index(drop=True)

In [ ]:
print(f'Versets bibliques chargés : {len(bible_df)}')
display(bible_df.head())

In [ ]:
def build_bible_records(bible_df):
    """Convertit le DataFrame biblique en enregistrements JSONL Passim."""
    records = []
    bible_text_by_ref = {}

    for _, row in bible_df.iterrows():
        book = str(row['book']).strip()
        chapter = str(row['chapter']).strip()
        verse = str(row['verse']).strip()
        reference = f"{book}_{chapter}:{verse}"
        text = str(row['latin']).strip()
        bible_id = f"bible::{reference}"

        records.append({
            'id': bible_id,
            'group': 'bible',
            'reference': reference,
            'book': book,
            'chapter': chapter,
            'verse': verse,
            'text': text,
        })
        bible_text_by_ref[reference] = text

    return records, bible_text_by_ref

## 5. Exécution de Passim

Le notebook impose un scénario unique :
- `source` = textes `.txt` ;
- `bible` = corpus de référence ;
- filtrage des paires : `source -> bible`.

Chaque exécution écrit aussi un fichier JSON de paramètres dans son dossier de run.

In [ ]:
# Chaque exécution est enregistrée dans un dossier distinct.
RUNS_ROOT = Path('passim_runs')
RUNS_ROOT.mkdir(exist_ok=True)

### 5.1 Les paramètres de Passim
Modifiez-les si nécessaire !


In [ ]:
PASSIM_N = 5 # taille des n-grammes en caractères
PASSIM_MIN_MATCH = 2 # nombre minimal de n-grammes communs
PASSIM_MIN_ALIGN = 10 # longueur minimale de l’alignement en caractères
PASSIM_GAP = 300 # écart maximal autorisé à l’intérieur d’un match
PASSIM_BEAM = 20 # largeur de la recherche
PASSIM_MAX_DF = 100 # seuil supérieur de document frequency
PASSIM_MIN_DF = 2 # seuil inférieur de document frequency
PASSIM_FILTERPAIRS = "group = 'source' AND group2 = 'bible'"

### 5.2 Création du dossier pour ce run

In [ ]:
RUN_ID = datetime.now().strftime('run_%Y%m%d_%H%M%S')
WORK_DIR = RUNS_ROOT / RUN_ID
INPUT_DIR = WORK_DIR / 'input'
OUTPUT_DIR = WORK_DIR / 'raw_output'
INPUT_DIR.mkdir(parents=True, exist_ok=True)

### 5.3 Génération de JSONL (input)

In [ ]:
bible_records, bible_text_by_ref = build_bible_records(bible_df)

bible_path = INPUT_DIR / 'bible_verses.jsonl'
with bible_path.open('w', encoding='utf-8') as f:
    for row in bible_records:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'Versets bibliques enregistrés : {bible_path}')
if bible_records:
    print('\nExemple :')
    print(json.dumps(bible_records[0], indent=2, ensure_ascii=False))

In [ ]:
merged_path = INPUT_DIR / 'passim_input.jsonl'
all_records = task_records + bible_records

with merged_path.open('w', encoding='utf-8') as f:
    for row in all_records:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'Entrée combinée : {merged_path}')
print(f'Enregistrements totaux : {len(all_records)}')
print(f'  source : {len(task_records)}')
print(f'  bible  : {len(bible_records)}')

### 5.4 Lancement de Passim

In [ ]:
task_path = INPUT_DIR / 'source_chunks.jsonl'
with task_path.open('w', encoding='utf-8') as f:
    for row in task_records:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

(WORK_DIR / 'documents_manifest.json').write_text(
    json.dumps([
        {
            'doc_id': doc['doc_id'],
            'source': doc['source'],
            'source_filename': doc['source_filename'],
            'source_origin': doc['source_origin'],
            'char_length': len(doc['text']),
        }
        for doc in YOUR_TEXTS
    ], indent=2, ensure_ascii=False),
    encoding='utf-8'
)

print(f'Dossier de cette exécution : {WORK_DIR}')
print(f'Chunks source enregistrés : {task_path}')
print(f'Manifeste des documents : {WORK_DIR / "documents_manifest.json"}')

In [ ]:
run_parameters = {
    'run_id': RUN_ID,
    'created_at': datetime.now().isoformat(),
    'txt_dir': str(TXT_DIR),
    'task_chunks': len(task_records),
    'bible_verses': len(bible_records),
    'chunking': {
        'mode': CHUNK_MODE,
        'sentences_per_chunk': SENTENCES_PER_CHUNK,
        'sentence_stride': SENTENCE_STRIDE,
        'char_chunk_size': CHUNK_SIZE,
        'char_chunk_overlap': CHUNK_OVERLAP,
        'min_chunk_chars': MIN_CHUNK_CHARS,
    },
    'passim': {
        'n': PASSIM_N,
        'min_match': PASSIM_MIN_MATCH,
        'min_align': PASSIM_MIN_ALIGN,
        'gap': PASSIM_GAP,
        'beam': PASSIM_BEAM,
        'max_df': PASSIM_MAX_DF,
        'min_df': PASSIM_MIN_DF,
        'filterpairs': PASSIM_FILTERPAIRS,
    },
}

(WORK_DIR / 'run_parameters.json').write_text(
    json.dumps(run_parameters, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

print(json.dumps(run_parameters, indent=2, ensure_ascii=False))

In [ ]:
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

command = [
    sys.executable, '-m', 'passim.seriatim',
    '--docwise',
    '--fields', 'group',
    '--filterpairs', PASSIM_FILTERPAIRS,
    '--output-format', 'json',
    '-n', str(PASSIM_N),
    '--min-match', str(PASSIM_MIN_MATCH),
    '--min-align', str(PASSIM_MIN_ALIGN),
    '--gap', str(PASSIM_GAP),
    '--beam', str(PASSIM_BEAM),
    '--maxDF', str(PASSIM_MAX_DF),
    '--minDF', str(PASSIM_MIN_DF),
    str(merged_path),
    str(OUTPUT_DIR),
]

env = os.environ.copy()
env.setdefault('PYSPARK_PYTHON', sys.executable)
env.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)

print('Commande exécutée :')
print(' '.join(command))
print('\nPatientez : Passim peut prendre de 30 à 90 secondes.')

completed = subprocess.run(command, capture_output=True, text=True, env=env)

(WORK_DIR / 'passim.stdout.log').write_text(completed.stdout or '', encoding='utf-8')
(WORK_DIR / 'passim.stderr.log').write_text(completed.stderr or '', encoding='utf-8')

if completed.returncode != 0:
    print('Passim a échoué.')
    print('\n--- STDERR (50 dernières lignes) ---')
    print('\n'.join((completed.stderr or '').splitlines()[-50:]))
else:
    print('Passim a terminé correctement.')
    print('\n--- STDOUT (20 dernières lignes) ---')
    print('\n'.join((completed.stdout or '').splitlines()[-20:]))
    print(f'\nRésultats bruts : {OUTPUT_DIR}')

## 6. Analyse des résultats et validation

In [ ]:
def load_passim_output(output_dir):
    out_json_dir = Path(output_dir) / 'out.json'
    if not out_json_dir.exists():
        raise FileNotFoundError(f'Sortie introuvable : {out_json_dir}')

    rows = []
    for path in sorted(out_json_dir.glob('part-*.json')):
        with path.open(encoding='utf-8') as f:
            rows.extend(json.loads(line) for line in f if line.strip())
    return rows


def parse_task_chunk_id(chunk_id):
    match = re.match(r'^source::(.+?)::chunk::(\d+)$', str(chunk_id))
    return (match.group(1), int(match.group(2))) if match else (None, None)


def parse_bible_id(bible_id):
    value = str(bible_id or '')
    return value.split('bible::', 1)[1] if value.startswith('bible::') else None


def normalize_for_matching(text):
    normalized_chars = []
    original_positions = []
    for index, char in enumerate(str(text or '')):
        if char.isalnum():
            normalized_chars.append(char.lower())
            original_positions.append(index)
    return ''.join(normalized_chars), original_positions


def locate_alignment_span(text, aligned, hint_start=None):
    haystack, positions = normalize_for_matching(text)
    needle, _ = normalize_for_matching(str(aligned or '').replace('-', ''))
    if not haystack or not needle:
        return 0, 0

    start_points = [0]
    if hint_start is not None and positions:
        start_points.insert(0, bisect_left(positions, max(0, int(hint_start))))

    start_index = -1
    for start_at in start_points:
        start_index = haystack.find(needle, start_at)
        if start_index >= 0:
            break
    if start_index < 0:
        return 0, 0

    end_index = start_index + len(needle) - 1
    return positions[start_index], positions[end_index] + 1


def reshape_results(raw_rows, chunk_meta, bible_text_by_ref):
    matches = []

    for doc in raw_rows:
        doc_id_field = str(doc.get('id', ''))
        for line in doc.get('lines', []) or []:
            for wit in line.get('wits', []) or []:
                wit_id = str(wit.get('id', ''))

                if doc_id_field.startswith('source::') and wit_id.startswith('bible::'):
                    chunk_id, bible_id = doc_id_field, wit_id
                    chunk_text = str(line.get('text', '') or doc.get('text', ''))
                    verse_text = bible_text_by_ref.get(parse_bible_id(bible_id), str(wit.get('text', '')))
                    chunk_hint, verse_hint = wit.get('begin2'), wit.get('begin')
                elif doc_id_field.startswith('bible::') and wit_id.startswith('source::'):
                    chunk_id, bible_id = wit_id, doc_id_field
                    chunk_text = chunk_meta.get(chunk_id, {}).get('text', str(wit.get('text', '')))
                    verse_text = bible_text_by_ref.get(parse_bible_id(bible_id), str(line.get('text', '') or doc.get('text', '')))
                    chunk_hint, verse_hint = wit.get('begin'), wit.get('begin2')
                else:
                    continue

                doc_id, chunk_index = parse_task_chunk_id(chunk_id)
                reference = parse_bible_id(bible_id)
                if not doc_id or not reference:
                    continue

                chunk_info = chunk_meta.get(chunk_id, {})
                alg_chunk = str(wit.get('alg', ''))
                alg_verse = str(wit.get('alg2', ''))
                chunk_align_start, chunk_align_end = locate_alignment_span(chunk_text, alg_chunk, chunk_hint)
                verse_align_start, verse_align_end = locate_alignment_span(verse_text, alg_verse, verse_hint)

                matches.append({
                    'doc_id': doc_id,
                    'source': chunk_info.get('source', doc_id),
                    'source_filename': chunk_info.get('source_filename'),
                    'source_origin': chunk_info.get('source_origin'),
                    'chunk_index': chunk_index,
                    'chunk_id': chunk_id,
                    'reference': reference,
                    'score': float(wit.get('matches', 0) or 0.0),
                    'chunk_text': chunk_text,
                    'verse_text': verse_text,
                    'chunk_align_start': chunk_align_start,
                    'chunk_align_end': chunk_align_end,
                    'verse_align_start': verse_align_start,
                    'verse_align_end': verse_align_end,
                    'alg_chunk': alg_chunk,
                    'alg_verse': alg_verse,
                })

    return sorted(matches, key=lambda row: (-row['score'], row['doc_id'], row['reference']))


def load_gold_annotations():
    gold_path = Path('data/csv/combined_biblical_citations_5_collections.csv')
    gold_df = pd.read_csv(gold_path)
    required_columns = {'source', 'doc_id', 'REF_book', 'REF_verse', 'Text'}
    missing_columns = required_columns - set(gold_df.columns)
    if missing_columns:
        raise ValueError(f"Colonnes manquantes dans le gold CSV : {sorted(missing_columns)}")
    return gold_path, gold_df


def evaluate_predictions(matches, gold_df):
    if not matches:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    best_pred = pd.DataFrame(matches).sort_values('score', ascending=False).drop_duplicates(['doc_id', 'reference']).copy()
    best_pred['doc_id'] = best_pred['doc_id'].astype(str).str.strip()

    gold = gold_df.copy()
    gold['doc_id'] = gold['doc_id'].astype(str).str.strip()
    gold['reference'] = gold['REF_book'].astype(str).str.strip() + '_' + gold['REF_verse'].astype(str).str.strip()

    pred_sets = best_pred.groupby('doc_id')['reference'].apply(set).to_dict()
    gold_sets = gold.groupby('doc_id')['reference'].apply(set).to_dict()
    source_by_doc = gold.groupby('doc_id')['source'].first().to_dict()

    rows = []
    for doc_id in sorted(set(pred_sets) | set(gold_sets)):
        predicted = pred_sets.get(doc_id, set())
        expected = gold_sets.get(doc_id, set())
        tp = len(predicted & expected)
        fp = len(predicted - expected)
        fn = len(expected - predicted)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        rows.append({
            'source': source_by_doc.get(doc_id),
            'doc_id': doc_id,
            'predicted_refs': len(predicted),
            'gold_refs': len(expected),
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'precision': precision,
            'recall': recall,
            'f1': 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0,
        })

    validation_by_doc_df = pd.DataFrame(rows).sort_values(
        ['f1', 'precision', 'recall', 'doc_id'],
        ascending=[False, False, False, True],
    )

    totals = validation_by_doc_df[['tp', 'fp', 'fn']].sum() if not validation_by_doc_df.empty else pd.Series({'tp': 0, 'fp': 0, 'fn': 0})
    precision = totals['tp'] / (totals['tp'] + totals['fp']) if (totals['tp'] + totals['fp']) else 0.0
    recall = totals['tp'] / (totals['tp'] + totals['fn']) if (totals['tp'] + totals['fn']) else 0.0
    validation_overall_df = pd.DataFrame([{
        'tp': int(totals['tp']),
        'fp': int(totals['fp']),
        'fn': int(totals['fn']),
        'precision': precision,
        'recall': recall,
        'f1': 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0,
    }])

    return best_pred, validation_by_doc_df, validation_overall_df

In [ ]:
raw_rows = load_passim_output(OUTPUT_DIR)
matches = reshape_results(raw_rows, chunk_meta, bible_text_by_ref)

print(f'Entrées brutes produites par Passim : {len(raw_rows)}')
print(f'Correspondances source -> bible : {len(matches)}')
if matches:
    print('\nTop 5 des meilleurs scores :')
    for match in matches[:5]:
        print(f"score={match['score']:.0f} | {match['doc_id']} | {match['reference']}")

gold_path, gold_df = load_gold_annotations()
best_predictions_df, validation_by_doc_df, validation_overall_df = evaluate_predictions(matches, gold_df)

if not best_predictions_df.empty:
    best_predictions_df.to_csv(WORK_DIR / 'best_predictions.csv', index=False)
if not validation_by_doc_df.empty:
    validation_by_doc_df.to_csv(WORK_DIR / 'validation_by_doc_id.csv', index=False)
if not validation_overall_df.empty:
    validation_overall_df.to_csv(WORK_DIR / 'validation_overall.csv', index=False)

print(f'Gold CSV utilisé : {gold_path}')

## 7. Tableaux de résultats et validation

In [ ]:
VIS_TABLE_ROWS = 50

if matches:
    best_table = pd.DataFrame(matches)[['doc_id', 'chunk_index', 'reference', 'score']]
    best_table = best_table.sort_values('score', ascending=False).drop_duplicates(['doc_id', 'reference'])
    best_table.to_csv(WORK_DIR / 'best_matches_preview.csv', index=False)
    print('Meilleures correspondances par document et verset :')
    display(best_table.head(VIS_TABLE_ROWS).reset_index(drop=True))
    if len(best_table) > VIS_TABLE_ROWS:
        print(f'Aperçu limité à {VIS_TABLE_ROWS} lignes.')
        print(f"Table complète enregistrée : {WORK_DIR / 'best_matches_preview.csv'}")
else:
    print("Aucune correspondance trouvée.")

if not validation_by_doc_df.empty:
    print('\nValidation par doc_id :')
    display(validation_by_doc_df.head(VIS_TABLE_ROWS).reset_index(drop=True))
    if len(validation_by_doc_df) > VIS_TABLE_ROWS:
        print(f'Aperçu validation limité à {VIS_TABLE_ROWS} lignes.')
        print(f"Table complète enregistrée : {WORK_DIR / 'validation_by_doc_id.csv'}")

if not validation_overall_df.empty:
    print('\nValidation globale :')
    display(validation_overall_df)

## 8. Comparaison de plusieurs runs Passim


In [ ]:
COMPARE_RUNS_ROOT = Path('passim_runs')
COMPARE_LAST_N_RUNS = 10

def flatten_run_parameters(data):
    flat = {}
    for key, value in data.items():
        if isinstance(value, dict):
            for subkey, subvalue in value.items():
                flat[f'{key}.{subkey}'] = subvalue
        else:
            flat[key] = value
    return flat


def load_run_summary(run_dir):
    params_path = run_dir / 'run_parameters.json'
    overall_path = run_dir / 'validation_overall.csv'
    by_doc_path = run_dir / 'validation_by_doc_id.csv'
    pred_path = run_dir / 'best_predictions.csv'

    if not params_path.exists():
        return None

    row = {'run_dir': str(run_dir)}
    row.update(flatten_run_parameters(json.loads(params_path.read_text(encoding='utf-8'))))

    if overall_path.exists():
        overall_df = pd.read_csv(overall_path)
        if not overall_df.empty:
            row.update({f'overall.{col}': overall_df.iloc[0][col] for col in overall_df.columns})

    if by_doc_path.exists():
        by_doc_df = pd.read_csv(by_doc_path)
        row['validation.docs'] = len(by_doc_df)
        row['validation.docs_with_tp'] = int((by_doc_df['tp'] > 0).sum()) if 'tp' in by_doc_df.columns else None

    if pred_path.exists():
        pred_df = pd.read_csv(pred_path)
        row['predictions.count'] = len(pred_df)

    return row


run_dirs = sorted(
    [p for p in COMPARE_RUNS_ROOT.glob('run_*') if p.is_dir()],
    key=lambda p: p.name,
    reverse=True,
)[:COMPARE_LAST_N_RUNS]

comparison_rows = [row for row in (load_run_summary(run_dir) for run_dir in run_dirs) if row]

if not comparison_rows:
    print('Aucun run comparable trouvé dans passim_runs/.')
else:
    comparison_df = pd.DataFrame(comparison_rows)
    preferred_columns = [
        'run_id',
        'created_at',
        'passim.n',
        'passim.min_match',
        'passim.min_align',
        'passim.gap',
        # 'passim.beam',
        # 'passim.max_df',
        # 'passim.min_df',
        # 'task_chunks',
        # 'bible_verses',
        'predictions.count',
        # 'validation.docs',
        # 'validation.docs_with_tp',
        'overall.tp',
        'overall.fp',
        'overall.fn',
        'overall.precision',
        'overall.recall',
        'overall.f1',
        #'run_dir',
    ]
    visible_columns = [col for col in preferred_columns if col in comparison_df.columns]
    comparison_df = comparison_df[visible_columns].sort_values(
        by=[col for col in ['overall.f1', 'overall.precision', 'overall.recall', 'created_at'] if col in visible_columns],
        ascending=[False, False, False, False][:len([col for col in ['overall.f1', 'overall.precision', 'overall.recall', 'created_at'] if col in visible_columns])]
    )

    comparison_path = COMPARE_RUNS_ROOT / 'runs_comparison.csv'
    comparison_df.to_csv(comparison_path, index=False)

    print(f'Runs comparés : {len(comparison_df)}')
    print(f'Table enregistrée : {comparison_path}')
    display(comparison_df.reset_index(drop=True))


## 9. Export HTML

In [ ]:
VIS_HTML_ROWS = 20
VIS_SNIPPET_PADDING = 140


def clip_text_for_display(text, start, end, padding=VIS_SNIPPET_PADDING):
    text = str(text or '')
    start = max(0, min(int(start or 0), len(text)))
    end = max(start, min(int(end or 0), len(text)))
    left = max(0, start - padding)
    right = min(len(text), end + padding)
    snippet = text[left:right]
    prefix = '…' if left > 0 else ''
    suffix = '…' if right < len(text) else ''
    return prefix + snippet + suffix, start - left + len(prefix), end - left + len(prefix)


def highlight_by_offsets(text, start, end, color):
    text = str(text or '')
    start = max(0, min(int(start or 0), len(text)))
    end = max(start, min(int(end or 0), len(text)))
    return (
        escape(text[:start])
        + f"<mark style='background:{color};padding:0 2px;border-radius:3px;font-weight:600'>"
        + escape(text[start:end])
        + '</mark>'
        + escape(text[end:])
    )


def render_html_table(matches, top_n=None):
    css = """
    <style>
      body{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin:16px}
      h2{color:#1a1a2e}
      p.meta{color:#566573;font-size:13px}
      table{border-collapse:collapse;width:100%;table-layout:fixed}
      th,td{border:1px solid #ddd;padding:8px 10px;vertical-align:top;font-size:13px}
      th{background:#f0f4ff;position:sticky;top:0;font-weight:600}
      td.mono{font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px}
      td.wrap{white-space:pre-wrap;word-break:break-word;line-height:1.5}
      tr:hover td{background:#fafafa}
      .score{color:#c0392b;font-weight:700}
      .dim{color:#888;font-size:11px}
    </style>
    """

    rows_html = []
    seen = set()

    for match in matches:
        key = (match['doc_id'], match['reference'])

        if key in seen:
            continue

        if top_n is not None and len(rows_html) >= top_n:
            break

        seen.add(key)

        chunk_snippet, chunk_start, chunk_end = clip_text_for_display(
            match['chunk_text'],
            match['chunk_align_start'],
            match['chunk_align_end']
        )

        verse_snippet, verse_start, verse_end = clip_text_for_display(
            match['verse_text'],
            match['verse_align_start'],
            match['verse_align_end']
        )

        rows_html.append(f"""
        <tr>
          <td class='mono'>{escape(match['doc_id'])}<br><span class='dim'>chunk {match['chunk_index']}</span></td>
          <td class='mono'>{escape(match['reference'])}</td>
          <td class='mono score'>{match['score']:.0f}</td>
          <td class='wrap'>{highlight_by_offsets(chunk_snippet, chunk_start, chunk_end, color='#fff176')}</td>
          <td class='wrap'>{highlight_by_offsets(verse_snippet, verse_start, verse_end, color='#c8f7c5')}</td>
        </tr>
        """)

    n_displayed = len(rows_html)
    total_unique = len({(m['doc_id'], m['reference']) for m in matches})

    if top_n is None:
        title = f"Résultats Passim : {n_displayed} paires"
    else:
        title = f"Résultats Passim : {n_displayed} premières paires sur {total_unique}"

    return (
        "<html><head><meta charset='utf-8'>" + css + "</head><body>"
        + f"<h2>{title}</h2>"
        + f"<p class='meta'>Run : {escape(RUN_ID)} | Dossier : {escape(str(WORK_DIR))}</p>"
        + "<table><tr><th style='width:170px'>doc_id</th><th style='width:110px'>Référence biblique</th><th style='width:70px'>Score</th><th>Texte source</th><th>Verset biblique de référence</th></tr>"
        + ''.join(rows_html)
        + "</table></body></html>"
    )


if matches:
    html_content_full = render_html_table(matches, top_n=None)
    html_content_preview = render_html_table(matches, top_n=VIS_HTML_ROWS)

    html_path = WORK_DIR / 'results.html'
    html_path.write_text(html_content_full, encoding='utf-8')

    display(HTML(html_content_preview))

    print(f'HTML enregistré : {html_path}')
    print(f"Aperçu HTML limité à {VIS_HTML_ROWS} lignes.")
else:
    print("Pas de correspondances à afficher en HTML.")

## 10. Meilleur alignement

In [ ]:
VIS_SNIPPET_PADDING = 140

if matches:
    best_match = matches[0]
    print('=' * 70)
    print(f"Meilleure correspondance : {best_match['doc_id']} -> {best_match['reference']}")
    print(f"Score : {best_match['score']:.0f}")
    print('=' * 70)
    print(f"\nTexte source [{best_match['chunk_align_start']}:{best_match['chunk_align_end']}] :")
    print(best_match['chunk_text'][max(0, best_match['chunk_align_start'] - VIS_SNIPPET_PADDING): best_match['chunk_align_end'] + VIS_SNIPPET_PADDING])
    print(f"\nVerset biblique [{best_match['verse_align_start']}:{best_match['verse_align_end']}] :")
    print(best_match['verse_text'][max(0, best_match['verse_align_start'] - VIS_SNIPPET_PADDING): best_match['verse_align_end'] + VIS_SNIPPET_PADDING])
    print('\nAlignement Passim :')
    print('Bible  :', best_match['alg_verse'])
    print('Source :', best_match['alg_chunk'])
else:
    print("Aucune correspondance à détailler.")